# Code summarization cookbook with Nano 3 on Fireworks AI

This notebook shows how to use the Nemotron Nano 3 model on Fireworks AI for code summarization.

You will learn how to:

- Summarize a single Python file
- Summarize all code cells inside a Jupyter notebook (`.ipynb`)
- Build reusable helper functions for code summarization workflows


## 1. Setup

You need:

- A Fireworks AI API key
- The `openai` Python client version 1.0 or later

Install dependencies if needed:

```bash
pip install --upgrade openai
```

Then set your key in the environment before running the notebook, for example:

```bash
export FIREWORKS_API_KEY="your_key_here"
```

In [1]:
# Install required packages using uv
! pip install -q fireworks-ai pandas matplotlib seaborn requests datasets huggingface-hub python-dotenv
! pip install openai

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
from pathlib import Path
import json
from typing import List, Tuple

from openai import OpenAI

FIREWORKS_API_KEY = os.getenv("FIREWORKS_API_KEY")

if not FIREWORKS_API_KEY:
    raise RuntimeError("FIREWORKS_API_KEY environment variable is not set")

client = OpenAI(
    api_key=FIREWORKS_API_KEY,
    base_url="https://api.fireworks.ai/inference/v1",
)

# Create a deployment for the Nano 3 model first, use the following command, and copy the deployed model name
# firectl -a <ACCOUNT-ID> create deployment accounts/fireworks/models/nemotron-nano-3-30b-a3b

# Deployed Nano 3 model on Fireworks AI
NANO3_MODEL = "accounts/pyroworks/deployedModels/nemotron-nano-3-30b-a3b-fmyofb76"

## 2. Core helper for summarization

This helper sends code to Nano 3 and asks for a concise engineer friendly summary.


In [ ]:
def summarize_code_snippet(code: str, *, filename: str = None, extra_instructions: str = None) -> str:
    """Summarize a block of Python code using Nano 3.

    Args:
        code: Source code as a string.
        filename: Optional filename for context in the prompt.
        extra_instructions: Optional extra instructions for the model,
            for example "focus on external API surface" or
            "explain for a junior engineer".
    """
    filename = filename or "snippet.py"

    system_msg = (
        "You are a senior Python engineer who explains code clearly and precisely. "
        "Summaries should be concise yet complete, written for other engineers. "
        "Highlight purpose, key functions or classes, inputs, outputs and any important design choices."
    )

    if extra_instructions:
        system_msg += " " + extra_instructions

    user_msg = f"Filename: {filename}\n\nCode:\n```python\n{code}\n```"

    response = client.chat.completions.create(
        model=NANO3_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
    )

    return response.choices[0].message.content.strip()


# Quick smoke test with a tiny snippet
test_code = """\
def add(a, b): 
    return a + b
    """
print(summarize_code_snippet(test_code, filename="example.py"))

Okay, the user wants me to act as a senior Python engineer to explain the given code example. Let me look at what they've provided.

The filename is example.py and the code is just a simple function called add that takes two parameters a and b and returns their sum. 

Hmm, this is a very basic example - probably meant as a starting point or illustration. I should explain it clearly but concisely since it's so simple. 

First, I need to identify the key components: the function definition, parameters, return statement, and the operation it performs. The purpose is straightforward - adding two numbers. 

I should highlight that this is a pure function with no side effects, takes two numeric inputs, and returns their sum. The inputs can be any types that support the + operator - not just integers but also floats, strings, etc. But since it's called "add", it's typically used with numbers.

Important design choices to mention: 
- Minimalist implementation (just one line of actual logic)
- 

## 3. Utility functions for loading and chunking code

Large files might not fit into a single prompt comfortably. A simple strategy is:

1. Split the file into chunks by line count.
2. Summarize each chunk.
3. Ask Nano 3 to synthesize a final summary from the chunk summaries.


In [7]:
def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf8")


def chunk_by_lines(text: str, max_lines: int = 200) -> List[str]:
    """Split text into chunks by line count.

    This is simple and works well for cookbook demos.
    """
    lines = text.splitlines()
    chunks = []
    for i in range(0, len(lines), max_lines):
        chunk = "\n".join(lines[i : i + max_lines])
        chunks.append(chunk)
    return chunks


def summarize_chunks(chunks: List[str], filename: str) -> Tuple[List[str], str]:
    """Summarize each chunk then synthesize a final summary.

    Returns the list of per chunk summaries and the final synthesized summary.
    """
    chunk_summaries = []
    for idx, chunk in enumerate(chunks):
        print(f"Summarizing chunk {idx + 1}/{len(chunks)}...")
        summary = summarize_code_snippet(
            chunk,
            filename=f"{filename} [chunk {idx + 1}]",
            extra_instructions="Focus only on what is new in this chunk and avoid repeating earlier chunks.",
        )
        chunk_summaries.append(summary)

    # Now synthesize one final summary from the per chunk summaries
    joined = "\n\n".join(
        f"Chunk {i + 1}: {s}" for i, s in enumerate(chunk_summaries)
    )

    system_msg = (
        "You are a senior engineer summarizing a large codebase from partial notes. "
        "Given chunk level summaries, produce one clear and structured overview of the file."
    )

    user_msg = (
        f"Here are summaries of different parts of the same file {filename}.\n\n" +
        joined +
        "\n\nWrite one high level summary that would help a new engineer onboard quickly."
    )

    response = client.chat.completions.create(
        model=NANO3_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
    )

    final_summary = response.choices[0].message.content.strip()
    return chunk_summaries, final_summary

## 4. Summarize a Python file

This helper accepts a path to a `.py` file and chooses between direct summarization and chunked summarization based on file size.


In [8]:
def summarize_python_file(path: str, *, max_direct_chars: int = 8000) -> None:
    path_obj = Path(path)
    if not path_obj.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    code = read_text_file(path_obj)
    filename = path_obj.name

    print(f"Loaded {filename} with {len(code)} characters")

    if len(code) <= max_direct_chars:
        print("Using direct summarization")
        summary = summarize_code_snippet(code, filename=filename)
        print("\n=== Summary ===\n")
        print(summary)
    else:
        print("File is large, using chunked summarization")
        chunks = chunk_by_lines(code, max_lines=200)
        chunk_summaries, final_summary = summarize_chunks(chunks, filename)

        print("\n=== Final summary ===\n")
        print(final_summary)

        print("\n=== Chunk level notes (optional) ===\n")
        for i, s in enumerate(chunk_summaries):
            print(f"--- Chunk {i + 1} ---")
            print(s)
            print()


# Example usage (update the path to point to a real file in your environment)
summarize_python_file("sample_data_pipeline.py")

Loaded sample_data_pipeline.py with 5864 characters
Using direct summarization

=== Summary ===

We need to provide concise yet complete summary for other engineers: purpose, key functions/classes, inputs/outputs, important design choices. No extra commentary, just summary. Ensure concise but complete. Provide bullet points maybe. No extra commentary.
</think>
**Purpose**  
A lightweight utility library that builds a chainable data‑processing pipeline for tabular records. It enables repeated transformations and validations on individual records, then aggregates results for batch export to CSV or JSON.

---

### Core Classes & Types  

| Name | Role | Key Methods / Attributes |
|------|------|---------------------------|
| **`ValidationResult`** | Encapsulates the outcome of a validation run. | `is_valid` (bool), `errors` (list), `warnings` (list); `__bool__` returns `is_valid`. |
| **`DataTransformer`** | Orchestrates a sequence of transformations and validators on records. | `__init__


**Resources:**
- [Fireworks AI Documentation](https://docs.fireworks.ai)
- [Firectl CLI Guide](https://docs.fireworks.ai/tools-sdks/firectl/firectl)
- [NVIDIA Nemotron Models](https://arxiv.org/abs/2504.03624)
- [GPQA Dataset Paper](https://arxiv.org/abs/2311.12022)
- [GPQA on Hugging Face](https://huggingface.co/datasets/Idavidrein/gpqa)

**Questions or Issues?**
- Visit [Fireworks AI Discord](https://discord.gg/fireworks-ai)
- Check [Fireworks AI Support](https://fireworks.ai/support)